# Scene Segmenter Training

Thin orchestrator notebook. Heavy lifting lives in `src/train_segmenter/`.

The configuration cell builds a `SegmenterPipelineConfig`. Everything else is a
single function call into the package, so reruns are cheap and deterministic.

In [ ]:
from pathlib import Path
import sys

candidate_src_dirs = [
    Path.cwd() / "src",
    Path.cwd() / "project" / "notebooks" / "do" / "src",
]
for src_dir in candidate_src_dirs:
    if src_dir.is_dir():
        parent = src_dir.parent.resolve()
        if str(parent) not in sys.path:
            sys.path.insert(0, str(parent))
        break
else:
    raise FileNotFoundError("Could not locate do/src directory for imports.")


from src.train_segmenter import (
    SegmenterPipelineConfig,
    initialize_segmenter_pipeline,
    plot_segmentation_predictions,
    plot_training_curves,
    run_segmenter_training,
)

CFG = SegmenterPipelineConfig(
    seed=42,
    image_size=256,
    val_split=0.35,
    cache_in_ram=True,
    num_workers=4,
    epochs=30,
    batch_size=4,
    learning_rate=1e-3,
    weight_decay=1e-4,
    use_amp=True,
    use_torch_compile=False,
    early_stopping_patience=8,
    log_every_batches=25,
    warm_start=False,
    preview_count=6,
)
CFG

## Initialize the pipeline

Builds the train/val pair lists, datasets and loaders, instantiates
`SceneUNetSmall`, the optimizer, scheduler and AMP scaler. Asserts the param
cap (R1) and that no training input lives under a `test` directory (R3).

In [ ]:
state = initialize_segmenter_pipeline(CFG)
state.keys()

## Train

Runs the BCE+Dice loop with verbose batch logging, saves the best checkpoint
on val IoU, and supports early stopping.

In [ ]:
state = run_segmenter_training(state)
{
    "best_val_iou": state.get("best_val_iou"),
    "best_epoch": state.get("best_epoch"),
    "training_seconds": state.get("training_seconds"),
}

## Training diagnostics

In [ ]:
plot_training_curves(state)

## Qualitative predictions on validation

In [ ]:
plot_segmentation_predictions(state, num_show=CFG.preview_count)